# 31 · Verify Site-1 Keys 0-4, Collect Site-2 (Tokyo<->NCSA) Keys 5-9

Two sections:

**A. Verify keys 0-4 (read-only, safe to re-run anytime).** Checks whether your
concern about overwriting is founded, without touching anything. Only offers
a rebuild path if verification actually finds a problem.

**B. Collect site-2 keys 5-9.** Site-1 now legitimately owns indices 0-9 --
so this section is deliberately built to NEVER write an unsuffixed filename
(e.g. `alice_sifted_bits_key5.json`) at any point, even transiently. Every
function takes an explicit `file_suffix` and writes directly to its final
destination (`..._key5_site2.json`), so there's no "write generic, rename
after" step for a bug to hide in -- that pattern is what caused every prior
incident in this project.

In [2]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))
RESULTS = PROJECT_DIR / "results"

import deploy_fabric as deploy
fablib = deploy.get_fablib()

# --- Physical provenance check: is this key ACTUALLY on this slice's Bob node? ---
def check_remote_provenance(slice_name, indices, suffix=""):
    slice_obj = fablib.get_slice(name=slice_name)
    bob = slice_obj.get_node("bob")
    results = {}
    for idx in indices:
        remote_name = f"alice_sifted_bits_key{idx}{suffix}.json"
        stdout, _ = bob.execute(f"test -f ~/qfabric/results/{remote_name} && cat ~/qfabric/results/{remote_name} || echo MISSING", quiet=True)
        if stdout.strip() == "MISSING":
            results[idx] = {"present": False, "n_bits": None}
            continue
        try:
            data = json.loads(stdout)
            results[idx] = {"present": True, "n_bits": len(data["alice_bits"])}
        except (json.JSONDecodeError, KeyError):
            results[idx] = {"present": True, "n_bits": "UNPARSEABLE"}
    return results

print("=== Site-1 slice 'qfabric-bb84-2': checking ALL 10 keys are physically there ===")
site1_remote = check_remote_provenance('qfabric-bb84-2', range(10))
meta1 = pd.read_csv(str(RESULTS / "key_pairs_metadata.csv")).set_index("index")
for idx, r in site1_remote.items():
    local_n = int(meta1.loc[idx, "n_bits"]) if idx in meta1.index else None
    match = "MATCH" if r["present"] and r["n_bits"] == local_n else "MISMATCH/MISSING"
    print(f"  key {idx}: remote_present={r['present']}, remote_n_bits={r['n_bits']}, local_n_bits={local_n} -> {match}")

print("\n=== Site-2 slice: checking its 5 keys are physically there ===")
SITE2_SLICE_NAME = 'qfabric-bb84-site2'  # confirm this matches your actual slice
site2_remote = check_remote_provenance(SITE2_SLICE_NAME, range(5), suffix="_site2")
meta2 = pd.read_csv(str(RESULTS / "key_pairs_metadata_site2.csv")).set_index("index")
for idx, r in site2_remote.items():
    local_n = int(meta2.loc[idx, "n_bits"]) if idx in meta2.index else None
    match = "MATCH" if r["present"] and r["n_bits"] == local_n else "MISMATCH/MISSING"
    print(f"  key {idx}: remote_present={r['present']}, remote_n_bits={r['n_bits']}, local_n_bits={local_n} -> {match}")

Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
CEPH Manager,https://ceph-mgr.fabric-testbed.net
Token File,/home/fabric/work/fabric_config/id_token.json
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
Bastion Host,bastion.fabric-testbed.net
Bastion Username,audreyf_0000527467
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub


=== Site-1 slice 'qfabric-bb84-2': checking ALL 10 keys are physically there ===
User: audreyf@illinois.edu bastion key is valid!
Configuration is valid
  key 0: remote_present=True, remote_n_bits=3504, local_n_bits=3504 -> MATCH
  key 1: remote_present=True, remote_n_bits=3409, local_n_bits=3409 -> MATCH
  key 2: remote_present=True, remote_n_bits=3360, local_n_bits=3360 -> MATCH
  key 3: remote_present=True, remote_n_bits=3431, local_n_bits=3431 -> MATCH
  key 4: remote_present=True, remote_n_bits=3456, local_n_bits=3456 -> MATCH
  key 5: remote_present=True, remote_n_bits=3455, local_n_bits=3455 -> MATCH
  key 6: remote_present=True, remote_n_bits=3446, local_n_bits=3446 -> MATCH
  key 7: remote_present=True, remote_n_bits=3388, local_n_bits=3388 -> MATCH
  key 8: remote_present=True, remote_n_bits=3430, local_n_bits=3430 -> MATCH
  key 9: remote_present=True, remote_n_bits=3351, local_n_bits=3351 -> MATCH

=== Site-2 slice: checking its 5 keys are physically there ===
  key 0: remo

In [3]:
# --- Timeline check: when was each key actually collected? ---
# File mtimes won't prove WHICH site, but they establish ORDER, which you
# can cross-reference against when in your own session history the
# Utah/Tokyo work happened.
import datetime
print("=== Local file modification times (collection order) ===")
for idx in range(10):
    p = RESULTS / f"alice_sifted_bits_key{idx}.json"
    if p.exists():
        mtime = datetime.datetime.fromtimestamp(p.stat().st_mtime)
        print(f"  site-1 key {idx}: {mtime}")
for idx in range(5):
    p = RESULTS / f"alice_sifted_bits_key{idx}_site2.json"
    if p.exists():
        mtime = datetime.datetime.fromtimestamp(p.stat().st_mtime)
        print(f"  site-2 key {idx}: {mtime}")

=== Local file modification times (collection order) ===
  site-1 key 0: 2026-09-18 23:59:58.091260
  site-1 key 1: 2026-09-18 23:59:58.358285
  site-1 key 2: 2026-09-18 23:59:58.608309
  site-1 key 3: 2026-09-18 23:59:58.857332
  site-1 key 4: 2026-09-18 23:59:59.105356
  site-1 key 5: 2026-09-18 05:48:30.255397
  site-1 key 6: 2026-09-18 05:49:26.352713
  site-1 key 7: 2026-09-18 05:50:23.960172
  site-1 key 8: 2026-09-18 05:51:21.850658
  site-1 key 9: 2026-09-18 05:52:18.785053
  site-2 key 0: 2026-09-18 23:29:13.000416
  site-2 key 1: 2026-09-18 23:30:22.864037
  site-2 key 2: 2026-09-18 23:31:26.272045
  site-2 key 3: 2026-09-18 23:32:30.414123
  site-2 key 4: 2026-09-18 23:33:35.367278


In [4]:
import pandas as pd

def sanity_check(metadata_path, label):
    df = pd.read_csv(metadata_path)
    print(f"=== {label} ({len(df)} keys) ===")
    problems = 0

    for _, row in df.iterrows():
        idx = int(row["index"])
        issues = []

        # n_bits should exactly equal m_sifted - k_pe (the PE split arithmetic)
        expected_n = row["m_sifted"] - row["k_pe"]
        if abs(row["n_bits"] - expected_n) > 0:
            issues.append(f"n_bits={row['n_bits']} != m_sifted-k_pe={expected_n}")

        # QBER must be a plausible physical value: >0 (a real channel always
        # has some noise) and well under BB84's own 11% abort threshold --
        # a run that exceeded 11% should have aborted, not produced a key
        if not (0 < row["qber"] < 0.11):
            issues.append(f"qber={row['qber']:.4f} outside plausible (0, 0.11) range")

        # k_pe (parameter-estimation sample) should be a sane fraction of
        # m_sifted -- collect_key_pairs used sample_fraction=0.1, so k_pe
        # should be close to 10% of m_sifted, not wildly off
        pe_fraction = row["k_pe"] / row["m_sifted"]
        if not (0.05 < pe_fraction < 0.15):
            issues.append(f"k_pe/m_sifted={pe_fraction:.3f}, expected ~0.10")

        # n_bits itself should be in a sane range for these experiments
        # (your keys have consistently run ~3300-3550; a value far outside
        # that isn't necessarily wrong, but is worth a second look)
        if not (2500 < row["n_bits"] < 4500):
            issues.append(f"n_bits={row['n_bits']} outside the ~3300-3550 range seen elsewhere")

        if issues:
            problems += 1
            print(f"  key {idx}: {'; '.join(issues)}")
        else:
            print(f"  key {idx}: OK  (qber={row['qber']:.4f}, n_bits={row['n_bits']}, k_pe={row['k_pe']})")

    print(f"\n{'All keys pass sanity checks' if problems == 0 else f'{problems} key(s) flagged above'}")
    return df

meta1 = sanity_check(str(RESULTS / "key_pairs_metadata.csv"), "Site-1")
print()
meta2 = sanity_check(str(RESULTS / "key_pairs_metadata_site2.csv"), "Site-2")

=== Site-1 (10 keys) ===
  key 0: OK  (qber=0.0128, n_bits=3504, k_pe=390)
  key 1: OK  (qber=0.0158, n_bits=3409, k_pe=379)
  key 2: OK  (qber=0.0027, n_bits=3360, k_pe=374)
  key 3: OK  (qber=0.0026, n_bits=3431, k_pe=382)
  key 4: OK  (qber=0.0052, n_bits=3456, k_pe=385)
  key 5: OK  (qber=0.0104, n_bits=3455, k_pe=384)
  key 6: OK  (qber=0.0235, n_bits=3446, k_pe=383)
  key 7: OK  (qber=0.0106, n_bits=3388, k_pe=377)
  key 8: OK  (qber=0.0105, n_bits=3430, k_pe=382)
  key 9: OK  (qber=0.0134, n_bits=3351, k_pe=373)

All keys pass sanity checks

=== Site-2 (5 keys) ===
  key 0: OK  (qber=0.0078, n_bits=3469, k_pe=386)
  key 1: OK  (qber=0.0026, n_bits=3411, k_pe=380)
  key 2: OK  (qber=0.0231, n_bits=3504, k_pe=390)
  key 3: OK  (qber=0.0161, n_bits=3339, k_pe=372)
  key 4: OK  (qber=0.0051, n_bits=3493, k_pe=389)

All keys pass sanity checks


## Section A -- Verify keys 0-4 (read-only)

Recomputes each key's true generation-portion QBER directly from the local
`_raw.json` files (if present) and compares against both the current
`key_pairs_metadata.csv` and the known-good values from prior recovery.
Changes nothing.

In [5]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))
RESULTS = PROJECT_DIR / "results"

import deploy_fabric as deploy
fablib = deploy.get_fablib()

# Known-good values, established via recovery earlier in this project.
# If Section A below finds a MISMATCH against these, something changed --
# investigate before doing anything else, don't just proceed.
KNOWN_GOOD_0_4 = {
    0: {"m_sifted": 3894, "k_pe": 390, "n_bits": 3504, "qber": 0.01282051282051282},
    1: {"m_sifted": 3788, "k_pe": 379, "n_bits": 3409, "qber": 0.0158311345646438},
    2: {"m_sifted": 3734, "k_pe": 374, "n_bits": 3360, "qber": 0.00267379679144385},
    3: {"m_sifted": 3813, "k_pe": 382, "n_bits": 3431, "qber": 0.002617801047120419},
    4: {"m_sifted": 3841, "k_pe": 385, "n_bits": 3456, "qber": 0.005194805194805195},
}


Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
CEPH Manager,https://ceph-mgr.fabric-testbed.net
Token File,/home/fabric/work/fabric_config/id_token.json
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
Bastion Host,bastion.fabric-testbed.net
Bastion Username,audreyf_0000527467
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub


In [6]:
# --- Check 1: does the local metadata CSV match the known-good table? ---
meta = pd.read_csv(str(RESULTS / "key_pairs_metadata.csv")).set_index("index")

print(f"{'key':<4}{'field':<10}{'metadata.csv':<20}{'known-good':<20}{'status'}")
any_mismatch = False
for idx, known in KNOWN_GOOD_0_4.items():
    if idx not in meta.index:
        print(f"{idx:<4}{'--':<10}{'MISSING FROM CSV':<20}{'':<20}MISMATCH")
        any_mismatch = True
        continue
    for field, known_val in known.items():
        csv_val = meta.loc[idx, field]
        ok = abs(csv_val - known_val) < 1e-9
        if not ok:
            any_mismatch = True
        print(f"{idx:<4}{field:<10}{csv_val:<20}{known_val:<20}{'OK' if ok else 'MISMATCH'}")

print(f"\n{'NO MISMATCHES -- metadata.csv matches known-good values' if not any_mismatch else '*** MISMATCH FOUND -- see Section A2 below ***'}")


key field     metadata.csv        known-good          status
0   m_sifted  3894                3894                OK
0   k_pe      390                 390                 OK
0   n_bits    3504                3504                OK
0   qber      0.0128205128205128  0.01282051282051282 OK
1   m_sifted  3788                3788                OK
1   k_pe      379                 379                 OK
1   n_bits    3409                3409                OK
1   qber      0.0158311345646438  0.0158311345646438  OK
2   m_sifted  3734                3734                OK
2   k_pe      374                 374                 OK
2   n_bits    3360                3360                OK
2   qber      0.0026737967914438  0.00267379679144385 OK
3   m_sifted  3813                3813                OK
3   k_pe      382                 382                 OK
3   n_bits    3431                3431                OK
3   qber      0.0026178010471204  0.002617801047120419OK
4   m_sifted  3841         

In [7]:
# --- Check 2: does the local metadata match what's ACTUALLY on the remote
# node right now? This catches the case where metadata.csv looks fine but
# the underlying key files were silently replaced by something else. ---
slice_obj = fablib.get_slice(name='qfabric-bb84-2')  # site-1 slice
bob = slice_obj.get_node("bob")

print(f"{'key':<4}{'remote file present':<22}{'remote n_bits':<16}{'local n_bits':<14}{'status'}")
for idx in range(5):
    remote_name = f"alice_sifted_bits_key{idx}.json"
    stdout, _ = bob.execute(f"test -f ~/qfabric/results/{remote_name} && cat ~/qfabric/results/{remote_name} || echo MISSING", quiet=True)
    if stdout.strip() == "MISSING":
        print(f"{idx:<4}{'MISSING':<22}{'--':<16}{'--':<14}MISMATCH")
        continue
    try:
        remote_data = json.loads(stdout)
        remote_n = len(remote_data["alice_bits"])
    except (json.JSONDecodeError, KeyError):
        print(f"{idx:<4}{'present but unparseable':<22}{'--':<16}{'--':<14}MISMATCH")
        continue
    local_n = int(meta.loc[idx, "n_bits"]) if idx in meta.index else None
    status = "OK" if remote_n == local_n else "MISMATCH"
    print(f"{idx:<4}{'present':<22}{remote_n:<16}{str(local_n):<14}{status}")


User: audreyf@illinois.edu bastion key is valid!
Configuration is valid
key remote file present   remote n_bits   local n_bits  status
0   present               3504            3504          OK
1   present               3409            3409          OK
2   present               3360            3360          OK
3   present               3431            3431          OK
4   present               3456            3456          OK


### A2 -- Only run this if either check above printed MISMATCH

Re-downloads keys 0-4 from Bob's remote node (unaffected by anything that
happened to local files) and rebuilds `key_pairs_metadata.csv` for just
those five rows. Safe to run even if nothing is wrong -- it will just
re-confirm the same values.

In [ ]:
alice = slice_obj.get_node("alice")
rebuilt_rows = []
for idx, known in KNOWN_GOOD_0_4.items():
    for who, remote_name in [("alice", f"alice_sifted_bits_key{idx}.json"),
                              ("bob", f"bob_sifted_bits_key{idx}.json")]:
        local_path = RESULTS / remote_name
        stdout, _ = bob.execute(f"cat ~/qfabric/results/{remote_name}", quiet=True)
        json.loads(stdout)  # sanity check it's real JSON before trusting it
        local_path.write_text(stdout)
    row = dict(known)
    row["index"] = idx
    row["alice_path"] = str(RESULTS / f"alice_sifted_bits_key{idx}.json")
    row["bob_path"] = str(RESULTS / f"bob_sifted_bits_key{idx}.json")
    rebuilt_rows.append(row)
    print(f"key {idx}: re-downloaded and verified")

# Merge into metadata: replace rows 0-4, keep everything else (5-9) untouched
existing = pd.read_csv(str(RESULTS / "key_pairs_metadata.csv"))
existing = existing[~existing["index"].isin(range(5))]
combined = pd.concat([pd.DataFrame(rebuilt_rows), existing], ignore_index=True).sort_values("index")
combined.to_csv(str(RESULTS / "key_pairs_metadata.csv"), index=False)
print(f"\nkey_pairs_metadata.csv rebuilt for keys 0-4, indices 5-9 left untouched. Now has {len(combined)} rows.")


## Section B -- Collect site-2 (Tokyo<->NCSA) keys 5-9

Every filename below is suffixed from the start -- `alice_sifted_bits_key5_site2.json`,
never `alice_sifted_bits_key5.json`. That file already belongs to site-1's
real key 5; this code never writes to it, not even briefly.

**Fill in your actual site-2 slice name below if it's not this.**

In [11]:
SITE2_SLICE_NAME = 'qfabric-bb84-site2'  # <-- confirm this matches your Tokyo<->NCSA slice
SITE2_FILE_SUFFIX = '_site2'
SITE2_METADATA_FILE = 'key_pairs_metadata_site2.csv'

slice_obj_2 = fablib.get_slice(name=SITE2_SLICE_NAME)
alice_2 = slice_obj_2.get_node("alice")
bob_2 = slice_obj_2.get_node("bob")

stdout, _ = bob_2.execute("ip -4 addr show | grep inet", quiet=True)
print(stdout)  # confirm bob_ip_2 below matches the dataplane address shown here
bob_ip_2 = "10.10.1.2"  # <-- confirm/replace based on the output above

print(f"\nsite-2 slice '{SITE2_SLICE_NAME}': {slice_obj_2.get_state()}")


    inet 127.0.0.1/8 scope host lo
    inet 10.20.4.48/23 metric 100 brd 10.20.5.255 scope global dynamic enp3s0


site-2 slice 'qfabric-bb84-site2': StableOK


In [ ]:
# import pandas as pd
# from qne.cascade.key import key_from_sifted_json
# from qne.cascade.parameter_estimation import split_pe_and_generation
#  
# def collect_key_pairs_safe(deploy, slice_obj, alice, bob, bob_ip, project_dir,
#                             start_index, n_keys, file_suffix, metadata_filename,
#                             scenario_path="validation/scenarios/fabric_1km.yml",
#                             pe_sample_fraction=0.1, pe_seed_base=500000, overwrite=False):
#     """Collision-safe by construction: every local AND remote filename is
#     built with file_suffix from the very first write, never as a rename
#     step after the fact. Reads/writes metadata_filename specifically, never
#     the bare key_pairs_metadata.csv, so site-1 and site-2 metadata can never
#     cross-contaminate either."""
#     target_indices = set(range(start_index, start_index + n_keys))
#     results_dir = project_dir / "results"

#     # --- Pre-flight: refuse if anything at this suffix+index already exists ---
#     existing_local = []
#     for i in target_indices:
#         for name in (f"alice_sifted_bits_key{i}{file_suffix}.json",
#                      f"bob_sifted_bits_key{i}{file_suffix}.json",
#                      f"alice_sifted_bits_key{i}{file_suffix}_raw.json",
#                      f"bob_sifted_bits_key{i}{file_suffix}_raw.json"):
#             p = results_dir / name
#             if p.exists():
#                 existing_local.append(str(p))

#     meta_path = results_dir / metadata_filename
#     existing_rows = set()
#     if meta_path.exists():
#         existing_meta = pd.read_csv(str(meta_path))
#         existing_rows = set(existing_meta["index"].astype(int)) & target_indices

#     if (existing_local or existing_rows) and not overwrite:
#         raise ValueError(
#             f"Refusing to collect indices {sorted(target_indices)} with suffix "
#             f"'{file_suffix}': would collide with {len(existing_local)} existing "
#             f"local file(s) and/or metadata rows {sorted(existing_rows)} in "
#             f"{metadata_filename}. Nothing touched, no FABRIC run started."
#         )

#     alice_mac = alice.get_interface(network_name="net_alice_switch").get_mac()
#     bob_mac = bob.get_interface(network_name="net_switch_bob").get_mac()
#     sw_alice_mac = slice_obj.get_node("switch").get_interface(network_name="net_alice_switch").get_mac()

#     key_pairs = []
#     for i in target_indices:
#         print(f"\n=== Collecting key pair {i}{file_suffix} ===")
#         deploy.run_bb84(slice_obj, scenario_path, alice_mac, bob_mac,
#                           sw_alice_mac=sw_alice_mac, bob_data_ip=bob_ip)

#         # Raw files: suffixed from the first write, no generic name ever touched
#         stdout, _ = alice.execute("cat ~/qfabric/results/alice_sifted_bits.json", quiet=True)
#         alice_raw_path = results_dir / f"alice_sifted_bits_key{i}{file_suffix}_raw.json"
#         alice_raw_path.write_text(stdout)

#         stdout, _ = bob.execute("cat ~/qfabric/results/bob_sifted_bits.json", quiet=True)
#         bob_raw_path = results_dir / f"bob_sifted_bits_key{i}{file_suffix}_raw.json"
#         bob_raw_path.write_text(stdout)

#         alice_key_full, alice_idx_full = key_from_sifted_json(str(alice_raw_path), "alice_bits")
#         bob_key_full, bob_idx_full = key_from_sifted_json(str(bob_raw_path), "bob_bits")

#         if alice_idx_full != bob_idx_full:
#             print(f"  key pair {i}: index mismatch -- skipping (raw files saved for inspection)")
#             continue

#         try:
#             split = split_pe_and_generation(alice_key_full, bob_key_full,
#                                              sample_fraction=pe_sample_fraction, seed=pe_seed_base + i)
#         except ValueError as e:
#             print(f"  key pair {i}: PE split failed ({e}) -- skipping")
#             continue

#         m, k, n = split["m"], split["k"], split["n"]
#         qber_i = split["qber"]
#         alice_key_i, bob_key_i = split["alice_gen"], split["bob_gen"]
#         print(f"  key pair {i}: m={m} sifted, k={k} PE sample, n={n} generation bits, qber={qber_i:.4f}")

#         alice_path = results_dir / f"alice_sifted_bits_key{i}{file_suffix}.json"
#         bob_path = results_dir / f"bob_sifted_bits_key{i}{file_suffix}.json"
#         gen_indices = split["gen_indices"]
#         alice_matching_indices_gen = [alice_idx_full[idx] for idx in gen_indices]

#         alice_path.write_text(json.dumps({"alice_bits": alice_key_i.bits.tolist(),
#                                            "matching_indices": alice_matching_indices_gen}))
#         bob_path.write_text(json.dumps({"bob_bits": bob_key_i.bits.tolist(),
#                                          "matching_indices": alice_matching_indices_gen}))

#         key_pairs.append({"index": i, "m_sifted": m, "k_pe": k, "n_bits": n, "qber": qber_i,
#                            "alice_path": str(alice_path), "bob_path": str(bob_path),
#                            "alice_raw_path": str(alice_raw_path), "bob_raw_path": str(bob_raw_path)})

#         # Remote upload: ALSO suffixed from the start -- site-2's bob node
#         # never even has a same-name file to collide with, but staying
#         # consistent avoids any future confusion either.
#         remote_alice_name = f"results/alice_sifted_bits_key{i}{file_suffix}.json"
#         remote_bob_name = f"results/bob_sifted_bits_key{i}{file_suffix}.json"
#         bob.upload_file(str(alice_path), f"qfabric/{remote_alice_name}")
#         bob.upload_file(str(bob_path), f"qfabric/{remote_bob_name}")

#     return pd.DataFrame(key_pairs)


# print("collect_key_pairs_safe defined")


In [ ]:
# existing_meta_2 = pd.read_csv(str(RESULTS / SITE2_METADATA_FILE)) if (RESULTS / SITE2_METADATA_FILE).exists() else pd.DataFrame()

# new_keys_df = collect_key_pairs_safe(
#     deploy, slice_obj_2, alice_2, bob_2, bob_ip_2, PROJECT_DIR,
#     start_index=5, n_keys=5,
#     file_suffix=SITE2_FILE_SUFFIX,
#     metadata_filename=SITE2_METADATA_FILE,
# )

# combined_meta_2 = pd.concat([existing_meta_2, new_keys_df], ignore_index=True)

# dupes = combined_meta_2["index"].duplicated()
# if dupes.any():
#     raise ValueError(f"Duplicate index(es): {combined_meta_2.loc[dupes, 'index'].tolist()} -- not writing.")

# combined_meta_2.to_csv(str(RESULTS / SITE2_METADATA_FILE), index=False)
# print(f"\n{SITE2_METADATA_FILE} now has {len(combined_meta_2)} keys (indices "
#       f"{sorted(combined_meta_2['index'].tolist())})")


## Section C -- Dependency check + dose-response for the new keys 5-9

Same `galois`/`randextract` check as before -- confirm it, don't assume it,
since this may be a slice that's never had the fix applied.

In [12]:
for node in [alice_2, bob_2]:
    check, _ = node.execute(
        "~/qfabric/.venv/bin/python3 -c 'from galois import GF2; import randextract; print(\"deps OK\")'",
        quiet=True,
    )
    if "deps OK" not in check:
        print(f"{node.get_name()}: MISSING DEPS, installing...")
        node.execute("cd ~/qfabric && source .venv/bin/activate && pip install --quiet -r requirements.txt", quiet=True)
        check, _ = node.execute(
            "~/qfabric/.venv/bin/python3 -c 'from galois import GF2; import randextract; print(\"deps OK\")'",
            quiet=True,
        )
    print(f"{node.get_name()}: {check.strip() or 'STILL MISSING -- stop and investigate'}")
    if "deps OK" not in check:
        raise RuntimeError(f"{node.get_name()} still missing deps, do not proceed to the sweep.")


alice: MISSING DEPS, installing...
alice: STILL MISSING -- stop and investigate


RuntimeError: alice still missing deps, do not proceed to the sweep.

In [13]:
# Does the venv even exist on this node?
stdout, _ = alice_2.execute("ls -la ~/qfabric/.venv/bin/ 2>&1 | head -5", quiet=True)
print("=== .venv/bin contents ===")
print(stdout)

# What does the raw install attempt actually say?
stdout, stderr = alice_2.execute(
    "cd ~/qfabric && source .venv/bin/activate && pip install -r requirements.txt 2>&1",
    quiet=True,
)
print("=== pip install output ===")
print(stdout)
print(stderr)

=== .venv/bin contents ===
ls: cannot access '/home/ubuntu/qfabric/.venv/bin/': No such file or directory

=== pip install output ===

bash: line 1: .venv/bin/activate: No such file or directory



In [10]:
from qne.cascade.sweep_utils import run_real_channel_trial

PROBS = [0.5, 0.3, 0.1, 0.05, 0.03, 0.01, 0.003, 0.001]
N_RUNS_PER_PROB = 20
CSV_SITE2_NEW = RESULTS / "sdc_site2_toeplitz_finalkey_doseresponse_keys5to9.csv"

key_pairs_df_2 = pd.read_csv(str(RESULTS / SITE2_METADATA_FILE))
new_keys_only = key_pairs_df_2[key_pairs_df_2["index"].isin(range(5, 10))]

all_rows = []
first_trial_checked = False

for _, meta_row in new_keys_only.iterrows():
    key_idx = int(meta_row["index"])
    k_i, qber_i = int(meta_row["k_pe"]), float(meta_row["qber"])
    a_remote = f"results/alice_sifted_bits_key{key_idx}{SITE2_FILE_SUFFIX}.json"
    b_remote = f"results/bob_sifted_bits_key{key_idx}{SITE2_FILE_SUFFIX}.json"
    print(f"\n=== site2 key{key_idx}: qber={qber_i:.4f} ===")

    for prob in PROBS:
        for fault_type, kwargs in [("toeplitz", {"toeplitz_prob": prob}),
                                    ("final_key", {"final_key_prob": prob})]:
            for run in range(N_RUNS_PER_PROB):
                seed = 42 + run
                result = run_real_channel_trial(bob_2, alice_2, bob_ip_2, qber_i, run, seed, k=k_i,
                                                   length_mode="asymptotic",
                                                   bob_key_json=b_remote, alice_key_json=a_remote,
                                                   **kwargs)
                result["fault_type"] = fault_type
                result["prob"] = prob
                result["key_index"] = key_idx
                result["qber"] = qber_i
                all_rows.append(result)

                if not first_trial_checked:
                    first_trial_checked = True
                    if result.get("error"):
                        raise RuntimeError(f"First trial failed: {result.get('error')!r}. "
                                            f"raw_stderr tail: {str(result.get('raw_stderr'))[-500:]}")
                    print("  first trial OK")

            pd.DataFrame(all_rows).to_csv(str(CSV_SITE2_NEW), index=False)

print(f"\nSaved {len(all_rows)} rows -> {CSV_SITE2_NEW}")
print("Combine with the existing keys-0-4 CSV (sdc_site2_toeplitz_finalkey_doseresponse.csv) "
      "for the full 10-key site-2 comparison.")


NameError: name 'SITE2_METADATA_FILE' is not defined

In [15]:
meta = pd.read_csv(str(RESULTS / "key_pairs_metadata.csv"))
new_keys = meta[meta["index"].isin(range(5, 10))]

for _, row in new_keys.iterrows():
    idx = int(row["index"])
    a_local = RESULTS / f"alice_sifted_bits_key{idx}.json"
    a_remote = f"results/alice_sifted_bits_key{idx}.json"
    alice.upload_file(str(a_local), f"qfabric/{a_remote}")
    print(f"key {idx}: uploaded alice-side file to alice's own node")

# sanity check before re-running the sweep
for idx in range(5, 10):
    check, _ = alice.execute(f"test -f ~/qfabric/results/alice_sifted_bits_key{idx}.json && echo present || echo MISSING", quiet=True)
    print(f"  key {idx} on alice's node: {check.strip()}")

key 5: uploaded alice-side file to alice's own node
key 6: uploaded alice-side file to alice's own node
key 7: uploaded alice-side file to alice's own node
key 8: uploaded alice-side file to alice's own node
key 9: uploaded alice-side file to alice's own node
  key 5 on alice's node: present
  key 6 on alice's node: present
  key 7 on alice's node: present
  key 8 on alice's node: present
  key 9 on alice's node: present


In [18]:
import sys, json
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))
RESULTS = PROJECT_DIR / "results"

import deploy_fabric as deploy
fablib = deploy.get_fablib()
slice_obj = fablib.get_slice(name='qfabric-bb84-2')  # site-1, already verified healthy
alice = slice_obj.get_node("alice")
bob = slice_obj.get_node("bob")
bob_ip = "10.10.1.2"

from qne.cascade.sweep_utils import run_real_channel_trial

PROBS = [0.5, 0.3, 0.1, 0.05, 0.03, 0.01, 0.003, 0.001]

# Sized down from the paper's original n=150/n=500 -- start here, confirm it
# runs cleanly end to end, THEN scale up if you want to match the original
# design exactly. At these reduced counts this is still ~4,800 trials.
N_TOEPLITZ_FINALKEY = 30   # paper used 150
N_RECONCILIATION = 100     # paper used 500

meta = pd.read_csv(str(RESULTS / "key_pairs_metadata.csv"))
new_keys = meta[meta["index"].isin(range(5, 10))]

all_rows = {"toeplitz": [], "final_key": [], "reconciliation": []}
first_trial_checked = {"toeplitz": False, "final_key": False, "reconciliation": False}

for _, meta_row in new_keys.iterrows():
    key_idx = int(meta_row["index"])
    k_i, qber_i = int(meta_row["k_pe"]), float(meta_row["qber"])
    a_remote = f"results/alice_sifted_bits_key{key_idx}.json"
    b_remote = f"results/bob_sifted_bits_key{key_idx}.json"
    print(f"\n=== site1 key{key_idx}: qber={qber_i:.4f} ===")

    for fault_type, n_runs, kwarg_name in [
        ("toeplitz", N_TOEPLITZ_FINALKEY, "toeplitz_prob"),
        ("final_key", N_TOEPLITZ_FINALKEY, "final_key_prob"),
        ("reconciliation", N_RECONCILIATION, "reconciliation_prob"),
    ]:
        for prob in PROBS:
            for run in range(n_runs):
                seed = 42 + run
                result = run_real_channel_trial(bob, alice, bob_ip, qber_i, run, seed, k=k_i,
                                                   length_mode="asymptotic",
                                                   bob_key_json=b_remote, alice_key_json=a_remote,
                                                   **{kwarg_name: prob})
                result["fault_type"] = fault_type
                result["prob"] = prob
                result["key_index"] = key_idx
                result["qber"] = qber_i
                all_rows[fault_type].append(result)

                if not first_trial_checked[fault_type]:
                    first_trial_checked[fault_type] = True
                    if result.get("error"):
                        raise RuntimeError(f"First {fault_type} trial failed: {result.get('error')!r}. "
                                            f"raw_stderr tail: {str(result.get('raw_stderr'))[-500:]}")
                    print(f"  first {fault_type} trial OK")

            # checkpoint after every (fault_type, prob) block
            for ft, rows in all_rows.items():
                if rows:
                    pd.DataFrame(rows).to_csv(str(RESULTS / f"sdc_realchannel_{ft}_keys5to9.csv"), index=False)

print("\nSaved:")
for ft in all_rows:
    print(f"  results/sdc_realchannel_{ft}_keys5to9.csv: {len(all_rows[ft])} rows")
print("\nCombine each with its keys-0-4 counterpart "
      "(sdc_realchannel_toeplitz_allkeys_asymptotic.csv, "
      "sdc_realchannel_finalkey_allkeys_asymptotic.csv, reconciliation_clean.csv) "
      "for the full 10-key versions of Figs 5-8 and the chi-square tests.")

Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
CEPH Manager,https://ceph-mgr.fabric-testbed.net
Token File,/home/fabric/work/fabric_config/id_token.json
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
Bastion Host,bastion.fabric-testbed.net
Bastion Username,audreyf_0000527467
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub


User: audreyf@illinois.edu bastion key is valid!
Configuration is valid

=== site1 key5: qber=0.0104 ===
  first toeplitz trial OK
  first final_key trial OK


TypeError: run_real_channel_trial() got an unexpected keyword argument 'reconciliation_prob'

In [2]:
import sys, json
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))
RESULTS = PROJECT_DIR / "results"

import deploy_fabric as deploy
fablib = deploy.get_fablib()
slice_obj = fablib.get_slice(name='qfabric-bb84-2')  # site-1, already verified healthy
alice = slice_obj.get_node("alice")
bob = slice_obj.get_node("bob")
bob_ip = "10.10.1.2"

from qne.cascade.sweep_utils import run_real_channel_trial, run_real_channel_reconciliation_trial

PROBS = [0.5, 0.3, 0.1, 0.05, 0.03, 0.01, 0.003, 0.001]
N_TOEPLITZ_FINALKEY = 30
N_RECONCILIATION = 30

meta = pd.read_csv(str(RESULTS / "key_pairs_metadata.csv"))
new_keys = meta[meta["index"].isin(range(5, 10))]

# --- Load whatever's already saved, so we can skip completed work ---
all_rows = {}
for ft in ["toeplitz", "final_key", "reconciliation"]:
    p = RESULTS / f"sdc_realchannel_{ft}_keys5to9.csv"
    all_rows[ft] = pd.read_csv(str(p)).to_dict("records") if p.exists() else []

def already_done(fault_type, key_idx, prob, n_needed):
    count = sum(1 for r in all_rows[fault_type] if r["key_index"] == key_idx and r["prob"] == prob)
    return count >= n_needed

first_trial_checked = {"toeplitz": False, "final_key": False, "reconciliation": False}

for _, meta_row in new_keys.iterrows():
    key_idx = int(meta_row["index"])
    k_i, qber_i = int(meta_row["k_pe"]), float(meta_row["qber"])
    a_remote = f"results/alice_sifted_bits_key{key_idx}.json"
    b_remote = f"results/bob_sifted_bits_key{key_idx}.json"
    print(f"\n=== site1 key{key_idx}: qber={qber_i:.4f} ===")

    for fault_type, kwarg_name in [("toeplitz", "toeplitz_prob"), ("final_key", "final_key_prob")]:
        for prob in PROBS:
            if already_done(fault_type, key_idx, prob, N_TOEPLITZ_FINALKEY):
                print(f"  {fault_type} key={key_idx} prob={prob}: already have >= {N_TOEPLITZ_FINALKEY} rows, skipping")
                continue
            for run in range(N_TOEPLITZ_FINALKEY):
                seed = 42 + run
                result = run_real_channel_trial(bob, alice, bob_ip, qber_i, run, seed, k=k_i,
                                                   length_mode="asymptotic",
                                                   bob_key_json=b_remote, alice_key_json=a_remote,
                                                   **{kwarg_name: prob})
                result["fault_type"] = fault_type
                result["prob"] = prob
                result["key_index"] = key_idx
                result["qber"] = qber_i
                all_rows[fault_type].append(result)

                if not first_trial_checked[fault_type]:
                    first_trial_checked[fault_type] = True
                    if result.get("error"):
                        raise RuntimeError(f"First {fault_type} trial failed: {result.get('error')!r}. "
                                            f"raw_stderr tail: {str(result.get('raw_stderr'))[-500:]}")
                    print(f"  first {fault_type} trial OK")

            pd.DataFrame(all_rows[fault_type]).to_csv(str(RESULTS / f"sdc_realchannel_{fault_type}_keys5to9.csv"), index=False)

    for prob in PROBS:
        if already_done("reconciliation", key_idx, prob, N_RECONCILIATION):
            print(f"  reconciliation key={key_idx} prob={prob}: already have >= {N_RECONCILIATION} rows, skipping")
            continue
        for run in range(N_RECONCILIATION):
            seed = 42 + run
            result = run_real_channel_reconciliation_trial(bob, alice, bob_ip, qber_i, run, seed,
                                                              reconciliation_prob=prob, k=k_i,
                                                              length_mode="asymptotic",
                                                              bob_key_json=b_remote, alice_key_json=a_remote)
            result["fault_type"] = "reconciliation"
            result["prob"] = prob
            result["key_index"] = key_idx
            result["qber"] = qber_i
            all_rows["reconciliation"].append(result)

            if not first_trial_checked["reconciliation"]:
                first_trial_checked["reconciliation"] = True
                if result.get("error"):
                    raise RuntimeError(f"First reconciliation trial failed: {result.get('error')!r}")
                print("  first reconciliation trial OK")

        pd.DataFrame(all_rows["reconciliation"]).to_csv(str(RESULTS / "sdc_realchannel_reconciliation_keys5to9.csv"), index=False)

print("\nFinal row counts:")
for ft in all_rows:
    print(f"  {ft}: {len(all_rows[ft])} rows")

Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
CEPH Manager,https://ceph-mgr.fabric-testbed.net
Token File,/home/fabric/work/fabric_config/id_token.json
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
Bastion Host,bastion.fabric-testbed.net
Bastion Username,audreyf_0000527467
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub


User: audreyf@illinois.edu bastion key is valid!
Configuration is valid


SliceNotFoundError: 'Unable to find slice "qfabric-bb84-2" for this project. Check slice name spelling and project id.'

In [20]:
stdout, _ = bob.execute("sudo lsof -i :5200 2>&1 || netstat -tlnp 2>&1 | grep 5200", quiet=True)
print("=== port 5200 ===")
print(stdout)

stdout, _ = bob.execute("ps aux | grep bob_cascade_driver", quiet=True)
print("=== bob_cascade_driver processes ===")
print(stdout)

ERROR:paramiko.transport:Secsh channel 0 open FAILED: No route to host: Connect failed
ERROR:paramiko.transport:Secsh channel 0 open FAILED: No route to host: Connect failed
ERROR:paramiko.transport:Secsh channel 0 open FAILED: No route to host: Connect failed


SSHError: SSH connection to node bob (2610:1e0:1700:206:f816:3eff:fe2a:dbff) failed after 3 attempts. Last error: ChannelException(2, 'Connect failed')

In [21]:
import time
for attempt in range(3):
    try:
        stdout, _ = bob.execute("echo test", quiet=True)
        print(f"attempt {attempt}: {stdout.strip()}")
        break
    except Exception as e:
        print(f"attempt {attempt} failed: {e}")
        time.sleep(10)

ERROR:paramiko.transport:Secsh channel 0 open FAILED: No route to host: Connect failed
ERROR:paramiko.transport:Secsh channel 0 open FAILED: No route to host: Connect failed
ERROR:paramiko.transport:Secsh channel 0 open FAILED: No route to host: Connect failed


attempt 0 failed: SSH connection to node bob (2610:1e0:1700:206:f816:3eff:fe2a:dbff) failed after 3 attempts. Last error: ChannelException(2, 'Connect failed')


ERROR:paramiko.transport:Secsh channel 0 open FAILED: No route to host: Connect failed
ERROR:paramiko.transport:Secsh channel 0 open FAILED: No route to host: Connect failed
ERROR:paramiko.transport:Secsh channel 0 open FAILED: No route to host: Connect failed


attempt 1 failed: SSH connection to node bob (2610:1e0:1700:206:f816:3eff:fe2a:dbff) failed after 3 attempts. Last error: ChannelException(2, 'Connect failed')


ERROR:paramiko.transport:Secsh channel 0 open FAILED: No route to host: Connect failed
ERROR:paramiko.transport:Secsh channel 0 open FAILED: No route to host: Connect failed
ERROR:paramiko.transport:Secsh channel 0 open FAILED: No route to host: Connect failed


attempt 2 failed: SSH connection to node bob (2610:1e0:1700:206:f816:3eff:fe2a:dbff) failed after 3 attempts. Last error: ChannelException(2, 'Connect failed')


In [23]:
# Is it just bob, or is alice unreachable too? If only bob fails, that's a
# site/host-specific problem, not slice expiration (which would hit both).
for node, name in [(alice, "alice"), (bob, "bob")]:
    try:
        stdout, _ = node.execute("echo test", quiet=True)
        print(f"{name}: reachable ({stdout.strip()})")
    except Exception as e:
        print(f"{name}: UNREACHABLE ({e})")

ERROR:paramiko.transport:Secsh channel 0 open FAILED: No route to host: Connect failed
ERROR:paramiko.transport:Secsh channel 0 open FAILED: No route to host: Connect failed
ERROR:paramiko.transport:Secsh channel 0 open FAILED: No route to host: Connect failed


alice: UNREACHABLE (SSH connection to node alice (2620:0:c80:1001:f816:3eff:fe69:24ed) failed after 3 attempts. Last error: ChannelException(2, 'Connect failed'))


ERROR:paramiko.transport:Secsh channel 0 open FAILED: No route to host: Connect failed
ERROR:paramiko.transport:Secsh channel 0 open FAILED: No route to host: Connect failed
ERROR:paramiko.transport:Secsh channel 0 open FAILED: No route to host: Connect failed


bob: UNREACHABLE (SSH connection to node bob (2610:1e0:1700:206:f816:3eff:fe2a:dbff) failed after 3 attempts. Last error: ChannelException(2, 'Connect failed'))


In [24]:
# Slice state and full details -- this is the thing to actually look at
try:
    state = slice_obj.get_state()
    print(f"site-1 slice state: {state}")
except Exception as e:
    print(f"Could not query slice state: {e}")

slice_obj.show()  # prints per-node status, IPs, reservation state -- read this fully

site-1 slice state: StableOK


ID,a2959fe6-7d36-4264-b81f-0449a1506af7
Name,qfabric-bb84-2
Lease Expiration (UTC),2026-09-23 02:53:18 +0000
Lease Start (UTC),2026-09-09 02:53:18 +0000
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
State,StableOK
Email,audreyf@illinois.edu
UserId,8616dd8c-5b61-45db-84bb-791c21e82a89


ID,a2959fe6-7d36-4264-b81f-0449a1506af7
Name,qfabric-bb84-2
Lease Expiration (UTC),2026-09-23 02:53:18 +0000
Lease Start (UTC),2026-09-09 02:53:18 +0000
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
State,StableOK
Email,audreyf@illinois.edu
UserId,8616dd8c-5b61-45db-84bb-791c21e82a89


In [25]:
# Check reachability first -- these commands need a working SSH connection,
# no point running them if the "no route to host" issue hasn't cleared yet.
for node, name in [(alice, "alice"), (bob, "bob")]:
    try:
        node.execute("echo test", quiet=True)
        print(f"{name}: reachable")
    except Exception as e:
        print(f"{name}: still unreachable -- {e}")

ERROR:paramiko.transport:Secsh channel 0 open FAILED: No route to host: Connect failed
ERROR:paramiko.transport:Secsh channel 0 open FAILED: No route to host: Connect failed
ERROR:paramiko.transport:Secsh channel 0 open FAILED: No route to host: Connect failed


alice: still unreachable -- SSH connection to node alice (2620:0:c80:1001:f816:3eff:fe69:24ed) failed after 3 attempts. Last error: ChannelException(2, 'Connect failed')


ERROR:paramiko.transport:Secsh channel 0 open FAILED: No route to host: Connect failed
ERROR:paramiko.transport:Secsh channel 0 open FAILED: No route to host: Connect failed
ERROR:paramiko.transport:Secsh channel 0 open FAILED: No route to host: Connect failed


bob: still unreachable -- SSH connection to node bob (2610:1e0:1700:206:f816:3eff:fe2a:dbff) failed after 3 attempts. Last error: ChannelException(2, 'Connect failed')


In [26]:
# Kill lingering processes and free the port on BOTH nodes -- the
# reconciliation trial spawns a process on each side (bob_cascade_driver.py
# on bob, alice_cascade_responder.py on alice), so a stuck run can leave
# orphans on either one.
for node, name in [(bob, "bob"), (alice, "alice")]:
    node.execute("pkill -f bob_cascade_driver.py; pkill -f alice_cascade_responder.py; sleep 1", quiet=True)
    print(f"{name}: killed any lingering driver/responder processes")

ERROR:paramiko.transport:Secsh channel 0 open FAILED: No route to host: Connect failed
ERROR:paramiko.transport:Secsh channel 0 open FAILED: No route to host: Connect failed
ERROR:paramiko.transport:Secsh channel 0 open FAILED: No route to host: Connect failed


SSHError: SSH connection to node bob (2610:1e0:1700:206:f816:3eff:fe2a:dbff) failed after 3 attempts. Last error: ChannelException(2, 'Connect failed')

In [ ]:
# Confirm port 5200 is actually free on bob (that's the one that binds it)
stdout, _ = bob.execute("sudo lsof -i :5200 2>&1 || echo 'port free'", quiet=True)
print("=== port 5200 on bob ===")
print(stdout)